# Solar Panel Detection & Energy Assessment
**PyGeoVision v2.0 — Notebook 08**

## Real-World Problem
A renewable energy company needs to inventory existing solar installations in
Zurich, Switzerland and estimate their annual energy generation potential.

```bash
pip install "pygeovision[geo,train]" geopandas shapely
```

In [ ]:
import pygeovision as pgv
import numpy as np, matplotlib.pyplot as plt, geopandas as gpd
from shapely.geometry import Point
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE_DIR = Path("./outputs/08_solar_panels")
BASE_DIR.mkdir(parents=True, exist_ok=True)

BBOX = (8.45, 47.30, 8.65, 47.45)   # Zurich, Switzerland
client = pgv.PyGeoVision()
print("Study area: Zurich, Switzerland")
print("Goal: Solar installation inventory + energy potential assessment")

## Step 1: Search and Download

In [ ]:
results = client.search(
    bbox=BBOX, date_range=("2024-05-01","2024-08-31"),
    providers=["planetary_computer"], cloud_cover_max=5, limit=3,
)
print(f"Scenes found: {len(results)}")
scene_path = None
if results:
    dl = client.download(results[:1], output_dir=BASE_DIR/"scenes",
                          post_process=["reproject:EPSG:32632","cog"])
    if dl and dl[0].success:
        scene_path = dl[0].path
        print(f"Downloaded: {Path(scene_path).name}")

## Step 2: Solar Panel Detection

In [ ]:
print("Solar Panel Detection Methods:")
print()
print("Method A — GeoAI SolarPanelDetector (recommended):")
if scene_path and Path(scene_path).exists():
    result = client.geoai.segment.solar_panels(
        scene_path, output_path=str(BASE_DIR/"solar_pred.tif"))
    print(f"  GeoAI result: {result}")
else:
    print("  result = client.geoai.segment.solar_panels(scene_path)")
print()
print("Method B — YOLOv8 object detection (good for utility-scale):")
print("  from pygeovision.models.detection.yolo import GeoYOLO")
print("  detector = GeoYOLO('yolov8-m', num_classes=1,")
print("                      class_names=['solar_farm'])")
print("  detections = detector.detect(scene_path, conf=0.40)")
print()
print("Method C — SegFormer segmentation (rooftop panels):")
print("  from pygeovision.models import get_model")
print("  from pygeovision.inference.tiled import TiledInference")
print("  model = get_model('segformer-b2', num_classes=2)")
print("  inf   = TiledInference(model, chip_size=512)")
print("  result = inf.infer(scene_path, 'solar_mask.tif')")

## Step 3: Energy Potential Estimation

In [ ]:
# Demo: synthetic solar panel inventory for Zurich
np.random.seed(42)
N_PANELS   = 1247
IRRADIANCE = 1150   # kWh/m2/year (Zurich annual average)
EFFICIENCY = 0.21   # 21% monocrystalline silicon
SYS_LOSS   = 0.80   # 80% system performance ratio (inverter, wiring, shading)

# Synthetic panel dataset
lat  = 47.30 + np.random.rand(N_PANELS) * 0.15
lon  = 8.45  + np.random.rand(N_PANELS) * 0.20
area = np.random.exponential(scale=22, size=N_PANELS) + 8   # m2
ptype = np.random.choice(
    ['residential','commercial','industrial'],
    size=N_PANELS, p=[0.68, 0.25, 0.07])

energy_kwh = area * IRRADIANCE * EFFICIENCY * SYS_LOSS

gdf = gpd.GeoDataFrame({
    'latitude': lat, 'longitude': lon,
    'area_m2': area, 'type': ptype,
    'energy_kwh_yr': energy_kwh,
    'capacity_kwp': area * EFFICIENCY,
}, crs="EPSG:4326")
gdf['geometry'] = [Point(lo, la) for lo, la in zip(lon, lat)]

print("=== Solar Energy Assessment — Zurich ===")
print(f"Installations detected  : {len(gdf):,}")
print(f"Total panel area        : {gdf.area_m2.sum()/1e4:.1f} ha")
print(f"Total capacity          : {gdf.capacity_kwp.sum()/1000:.1f} MWp")
print(f"Annual energy production: {gdf.energy_kwh_yr.sum()/1e6:.1f} GWh/year")
print(f"CO2 avoided (230g/kWh)  : {gdf.energy_kwh_yr.sum()*230/1e9:.1f} kt CO2/year")
print()
for pt in ['residential','commercial','industrial']:
    sub = gdf[gdf.type == pt]
    print(f"  {pt:<14}: {len(sub):4d} installations  "
          f"{sub.energy_kwh_yr.sum()/1e6:.1f} GWh/yr")

## Step 4: Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = {'residential':'#F39C12','commercial':'#2980B9','industrial':'#8E44AD'}
for ptype, color in colors.items():
    sub = gdf[gdf.type == ptype]
    axes[0].scatter(sub.longitude, sub.latitude, c=color, alpha=0.55,
                     s=sub.area_m2*0.3, label=ptype)
axes[0].set_title(f"Solar Installations — Zurich\n(size = panel area)",
                   fontsize=11, fontweight='bold')
axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")
axes[0].legend(fontsize=9)

axes[1].hist(gdf.area_m2.clip(upper=200), bins=30,
              color='#F39C12', edgecolor='white', alpha=0.85, log=True)
axes[1].set_xlabel("Panel Area (m2)"); axes[1].set_ylabel("Count (log scale)")
axes[1].set_title("Installation Size Distribution", fontsize=11, fontweight='bold')

energy_by_type = gdf.groupby('type')['energy_kwh_yr'].sum() / 1e6
axes[2].bar(energy_by_type.index, energy_by_type.values,
             color=[colors[k] for k in energy_by_type.index])
axes[2].set_ylabel("Annual Energy (GWh)")
axes[2].set_title("Energy Production by Type", fontsize=11, fontweight='bold')
for i, (k, v) in enumerate(energy_by_type.items()):
    axes[2].text(i, v+0.05, f"{v:.1f} GWh", ha='center', fontsize=10, fontweight='bold')

plt.suptitle("Solar Panel Detection & Energy Assessment — Zurich, Switzerland",
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR/"solar_assessment.png", dpi=150, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
print("=" * 60)
print("NOTEBOOK 08 — Solar Panel Detection & Energy Assessment")
print("=" * 60)
print(f"City           : Zurich, Switzerland")
print(f"Installations  : {len(gdf):,}")
print(f"Total capacity : {gdf.capacity_kwp.sum()/1000:.1f} MWp")
print(f"Annual energy  : {gdf.energy_kwh_yr.sum()/1e6:.1f} GWh/year")
print()
print("Next: 09_disaster_damage_assessment.ipynb")